# Features extraction code

## Requirements

In [1]:
!pip3 install pandas openpyxl numpy

## Imports

In [2]:
import pandas as pd
import openpyxl
import json
import statistics
import numpy as np

## Dataframe creation and general variables

In [3]:
features_df = pd.read_excel("../extracted_features/extracted_features.xlsx")

data_path = "../data/"
valid_participants = features_df['participant_id']

features_df

,participant_id,gender,age,perceived_topic_intimacy_c1,perceived_topic_intimacy_c2,personal_discomfort_c1,personal_discomfort_c2,revealing_personal_information_c1,revealing_personal_information_c2
0,4,M,23,5,4,1.571,1.000,1.25,1.00
1,5,M,20,3,4,1.143,2.429,1.25,2.00
2,6,F,27,4,3,1.143,1.286,1.00,2.75
3,7,F,21,1,3,1.000,1.000,1.75,1.75
4,9,M,26,4,2,1.714,1.286,2.00,1.50
5,11,F,28,2,4,1.000,1.286,1.00,2.25
6,12,F,19,2,5,1.143,2.571,3.25,5.00
7,13,F,19,2,5,1.143,2.429,1.50,3.25
8,14,F,24,4,3,1.000,1.000,3.00,3.00
9,15,M,29,3,4,1.143,2.143,1.50,1.00


## General features

### Activity time

In [4]:
for condition in ["c1", "c2"]:
    activity_time_list = []
    for participant in valid_participants:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        activity_time_list.append(round(data["activity_time"], 2))
    new_column_name = "activity_time_" + condition
    features_df[new_column_name] = activity_time_list

## Drawing dynamics features

### Stroke count

In [5]:
for condition in ["c1", "c2"]:
    stroke_count_list = []
    for participant in valid_participants:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)
        stroke_count_list.append(len(data["strokes"]) + len(data["undone_erased_strokes"]))
    new_column_name = "stroke_count_" + condition
    features_df[new_column_name] = stroke_count_list

,participant_id,gender,age,perceived_topic_intimacy_c1,perceived_topic_intimacy_c2,personal_discomfort_c1,personal_discomfort_c2,revealing_personal_information_c1,revealing_personal_information_c2,activity_time_c1,activity_time_c2,stroke_count_c1,stroke_count_c2
0,4,M,23,5,4,1.571,1.000,1.25,1.00,1209.19,1118.29,464,311
1,5,M,20,3,4,1.143,2.429,1.25,2.00,976.01,1007.23,483,318
2,6,F,27,4,3,1.143,1.286,1.00,2.75,849.10,556.19,115,137
3,7,F,21,1,3,1.000,1.000,1.75,1.75,679.11,508.09,157,105
4,9,M,26,4,2,1.714,1.286,2.00,1.50,796.32,564.09,106,83
5,11,F,28,2,4,1.000,1.286,1.00,2.25,967.74,959.02,355,472
6,12,F,19,2,5,1.143,2.571,3.25,5.00,857.95,822.00,314,250
7,13,F,19,2,5,1.143,2.429,1.50,3.25,580.20,718.35,102,107
8,14,F,24,4,3,1.000,1.000,3.00,3.00,877.48,914.74,360,239
9,15,M,29,3,4,1.143,2.143,1.50,1.00,850.13,654.78,177,273


### Stroke length (total, mean, std)

In [6]:
def compute_stroke_length(stroke_points):
        point_coordinates = np.array(stroke_points)

        # Differences between consecutive points
        diffs = np.diff(point_coordinates, axis=0)

        # Euclidian distances between consecutive points
        euclidian_distances = np.sqrt((diffs ** 2).sum(axis=1))

        # Sum all the distances between each consecutive points of the stroke
        total_stroke_length = euclidian_distances.sum()

        return total_stroke_length

In [7]:
for condition in ["c1", "c2"]:
    stroke_total_length_list = []
    stroke_mean_length_list = []
    stroke_std_length_list = []
    
    for participant in valid_participants:
        stroke_file_path = data_path + "p" + str(participant) + "/drawing_data/strokes_" + condition + ".json"
        with open(stroke_file_path, "r") as file:
            data = json.load(file)

        # Extract length of each stroke of participant
        stroke_length_array = []
        for stroke in (data["strokes"] + data["undone_erased_strokes"]):
            stroke_length_array.append(compute_stroke_length(stroke["points"]))

        # Compute total, mean, and std stroke length
        stroke_total_length_list.append(round(sum(stroke_length_array), 2))
        stroke_mean_length_list.append(round(statistics.mean(stroke_length_array), 2))
        stroke_std_length_list.append(round(statistics.stdev(stroke_length_array), 2))
        
    # Add column to dataframe
    stroke_total_length_name = "stroke_total_length_" + condition
    stroke_mean_length_name = "stroke_mean_length_" + condition
    stroke_std_length_name = "stroke_std_length_" + condition
    features_df[stroke_total_length_name] = stroke_total_length_list
    features_df[stroke_mean_length_name] = stroke_mean_length_list
    features_df[stroke_std_length_name] = stroke_std_length_list

features_df

,participant_id,gender,age,perceived_topic_intimacy_c1,perceived_topic_intimacy_c2,personal_discomfort_c1,personal_discomfort_c2,revealing_personal_information_c1,revealing_personal_information_c2,activity_time_c1,activity_time_c2,stroke_count_c1,stroke_count_c2,stroke_total_length_c1,stroke_mean_length_c1,stroke_std_length_c1,stroke_total_length_c2,stroke_mean_length_c2,stroke_std_length_c2
0,4,M,23,5,4,1.571,1.000,1.25,1.00,1209.19,1118.29,464,311,32406.17,69.84,85.95,31294.04,100.62,171.23
1,5,M,20,3,4,1.143,2.429,1.25,2.00,976.01,1007.23,483,318,251620.84,520.95,1473.13,207701.98,653.15,2489.35
2,6,F,27,4,3,1.143,1.286,1.00,2.75,849.10,556.19,115,137,29558.12,257.03,398.05,15112.46,110.31,128.92
3,7,F,21,1,3,1.000,1.000,1.75,1.75,679.11,508.09,157,105,32312.35,205.81,349.62,18139.80,172.76,335.99
4,9,M,26,4,2,1.714,1.286,2.00,1.50,796.32,564.09,106,83,149419.08,1409.61,8492.46,31562.58,380.27,704.23
5,11,F,28,2,4,1.000,1.286,1.00,2.25,967.74,959.02,355,472,28442.45,80.12,83.10,24608.77,52.14,49.55
6,12,F,19,2,5,1.143,2.571,3.25,5.00,857.95,822.00,314,250,25720.26,81.91,150.50,101484.88,405.94,1375.51
7,13,F,19,2,5,1.143,2.429,1.50,3.25,580.20,718.35,102,107,7907.19,77.52,88.66,13350.23,124.77,169.54
8,14,F,24,4,3,1.000,1.000,3.00,3.00,877.48,914.74,360,239,30949.27,85.97,119.74,39453.10,165.08,237.53
9,15,M,29,3,4,1.143,2.143,1.50,1.00,850.13,654.78,177,273,52409.46,296.10,746.82,77535.20,284.01,634.22


### Stroke speed and acceleration (mean and std)

### Stroke pressure (mean, std, slope over time?)

### Temporal drawing behavior

### Editing behavior

## Drawing production features

## Speech dynamics features

### Pitch (mean, std, percentil-based range)

### Loudness (mean, std)

### Voice quality (Jitter, Shimmer, HNR)

### Temporal speaking behavior

## Speech production features (transcript)

## Display and save final features dataframe